In [1]:
import os, glob

# Check working directory for any saved files
print("=== /kaggle/working/ ===")
for f in glob.glob("/kaggle/working/**/*", recursive=True):
    size = os.path.getsize(f) / (1024*1024)
    print(f"  {f}  ({size:.1f} MB)")

=== /kaggle/working/ ===
  /kaggle/working/__notebook__.ipynb  (0.0 MB)


In [2]:
# ==============================================================================
# FAST RETRAIN — Optimized for ~50 minutes / 13 epochs
# Saves checkpoint immediately every time val acc improves
# Includes: EMA, CutMix, 4-TTA, gradient clipping, stratified split
# ==============================================================================

import os, glob, random
import numpy as np
import pandas as pd
from PIL import Image
from collections import Counter

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms, models
from torch.optim.swa_utils import AveragedModel, get_ema_multi_avg_fn
from sklearn.model_selection import train_test_split
from tqdm import tqdm

# ══════════════════════════════════════════════════════════════════
#  1. SEED & DEVICE
# ══════════════════════════════════════════════════════════════════
SEED = 42
def seed_everything(seed):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

seed_everything(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"🖥️  Device: {device}")

# ══════════════════════════════════════════════════════════════════
#  2. PATHS
# ══════════════════════════════════════════════════════════════════
def find_data_paths():
    for root, dirs, _ in os.walk('/kaggle/input'):
        if 'train' in dirs and 'test' in dirs:
            return os.path.join(root, 'train'), os.path.join(root, 'test')
    return None, None

DATA_DIR, TEST_DIR = find_data_paths()
SAVE_PATH = "/kaggle/working/best_model.pth"
print(f"📂 Train: {DATA_DIR}")
print(f"📂 Test:  {TEST_DIR}")

# ══════════════════════════════════════════════════════════════════
#  3. CONFIG — tuned for ~13 epochs / 50 minutes
# ══════════════════════════════════════════════════════════════════
IMG_SIZE   = 288
BATCH_SIZE = 16
EPOCHS     = 13
MEAN       = [0.485, 0.456, 0.406]
STD        = [0.229, 0.224, 0.225]

# ══════════════════════════════════════════════════════════════════
#  4. TRANSFORMS
# ══════════════════════════════════════════════════════════════════
train_transforms = transforms.Compose([
    transforms.RandomResizedCrop(IMG_SIZE, scale=(0.6, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(0.2, 0.2, 0.2),
    transforms.RandAugment(num_ops=2, magnitude=9),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])

val_transforms = transforms.Compose([
    transforms.Resize(320),
    transforms.CenterCrop(IMG_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])

# 4 TTA transforms for inference
t_flip = transforms.Compose([
    transforms.Resize(320),
    transforms.CenterCrop(IMG_SIZE),
    transforms.RandomHorizontalFlip(p=1.0),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])
t_large = transforms.Compose([
    transforms.Resize(360),
    transforms.CenterCrop(IMG_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])
t_jitter = transforms.Compose([
    transforms.Resize(320),
    transforms.CenterCrop(IMG_SIZE),
    transforms.ColorJitter(brightness=0.1, contrast=0.1),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])
TTA_TRANSFORMS = [val_transforms, t_flip, t_large, t_jitter]

# ══════════════════════════════════════════════════════════════════
#  5. MODEL
# ══════════════════════════════════════════════════════════════════
class CustomConvNeXt(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        base = models.convnext_base(weights=models.ConvNeXt_Base_Weights.IMAGENET1K_V1)

        self.block0 = nn.Sequential(*base.features[0:2])
        self.block1 = nn.Sequential(*base.features[2:4])
        self.block2 = nn.Sequential(*base.features[4:6])
        self.block3 = nn.Sequential(*base.features[6:8])

        in_features = base.classifier[2].in_features

        self.head = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            base.classifier[0],
            nn.Flatten(),
            nn.Linear(in_features, 1024),
            nn.BatchNorm1d(1024),
            nn.GELU(),
            nn.Dropout(0.4),
            nn.Linear(1024, num_classes)
        )

    def forward(self, x):
        return self.head(self.block3(self.block2(self.block1(self.block0(x)))))

# ══════════════════════════════════════════════════════════════════
#  6. MIXUP & CUTMIX
# ══════════════════════════════════════════════════════════════════
def mixup_data(x, y, alpha=0.2):
    lam = torch.distributions.Beta(alpha, alpha).sample().item()
    idx = torch.randperm(x.size(0)).to(device)
    return lam * x + (1 - lam) * x[idx], y, y[idx], lam

def cutmix_data(x, y, alpha=1.0):
    lam = torch.distributions.Beta(alpha, alpha).sample().item()
    idx = torch.randperm(x.size(0)).to(device)
    W, H = x.size(3), x.size(2)
    cut_w = int(W * (1 - lam) ** 0.5)
    cut_h = int(H * (1 - lam) ** 0.5)
    cx = torch.randint(W, (1,)).item()
    cy = torch.randint(H, (1,)).item()
    x1, x2 = max(cx - cut_w // 2, 0), min(cx + cut_w // 2, W)
    y1, y2 = max(cy - cut_h // 2, 0), min(cy + cut_h // 2, H)
    x[:, :, y1:y2, x1:x2] = x[idx, :, y1:y2, x1:x2]
    lam = 1 - (x2 - x1) * (y2 - y1) / (W * H)
    return x, y, y[idx], lam

def mixed_loss(criterion, pred, y_a, y_b, lam):
    return lam * criterion(pred, y_a) + (1 - lam) * criterion(pred, y_b)

# ══════════════════════════════════════════════════════════════════
#  7. MAIN
# ══════════════════════════════════════════════════════════════════
def run():
    # Dataset
    full_ds     = datasets.ImageFolder(DATA_DIR)
    num_classes = len(full_ds.classes)
    print(f"📊 {num_classes} classes | {len(full_ds)} total images")

    label_counts   = Counter(full_ds.targets)
    dominant_class = label_counts.most_common(1)[0][0]

    train_idx, val_idx = train_test_split(
        np.arange(len(full_ds)),
        test_size=0.10,
        stratify=full_ds.targets,
        random_state=SEED
    )

    train_ds = Subset(datasets.ImageFolder(DATA_DIR, train_transforms), train_idx)
    val_ds   = Subset(datasets.ImageFolder(DATA_DIR, val_transforms),   val_idx)

    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                              num_workers=2, pin_memory=True)
    val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False,
                              num_workers=2, pin_memory=True)

    print(f"   Train: {len(train_ds)} | Val: {len(val_ds)}")

    # Model + EMA
    model     = CustomConvNeXt(num_classes).to(device)
    ema_model = AveragedModel(model, multi_avg_fn=get_ema_multi_avg_fn(0.999))

    # LLRD optimizer
    optimizer = torch.optim.AdamW([
        {'params': model.block0.parameters(), 'lr': 1e-5},
        {'params': model.block1.parameters(), 'lr': 1e-5},
        {'params': model.block2.parameters(), 'lr': 2e-5},
        {'params': model.block3.parameters(), 'lr': 2e-5},
        {'params': model.head.parameters(),   'lr': 1e-4},
    ], weight_decay=0.05)

    # Cosine restarts every 5 epochs
    scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(optimizer, T_0=5)
    criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
    scaler    = torch.amp.GradScaler('cuda')
    best_acc  = 0.0

    # ── Training ──────────────────────────────────────────────────
    print(f"\n🔥 Training {EPOCHS} epochs (~{EPOCHS * 4} min)...")
    for epoch in range(EPOCHS):
        model.train()
        total_loss, total = 0.0, 0

        for imgs, lbls in tqdm(train_loader, desc=f"Ep {epoch+1:02d}/{EPOCHS}"):
            imgs, lbls = imgs.to(device), lbls.to(device)

            r = torch.rand(1).item()
            if r < 0.33:
                imgs, y_a, y_b, lam = mixup_data(imgs, lbls)
                with torch.amp.autocast('cuda'):
                    loss = mixed_loss(criterion, model(imgs), y_a, y_b, lam)
            elif r < 0.66:
                imgs, y_a, y_b, lam = cutmix_data(imgs, lbls)
                with torch.amp.autocast('cuda'):
                    loss = mixed_loss(criterion, model(imgs), y_a, y_b, lam)
            else:
                with torch.amp.autocast('cuda'):
                    loss = criterion(model(imgs), lbls)

            optimizer.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            scaler.step(optimizer)
            scaler.update()
            ema_model.update_parameters(model)

            total_loss += loss.item() * imgs.size(0)
            total      += imgs.size(0)

        scheduler.step()

        # Validation
        ema_model.eval()
        correct, total_val = 0, 0
        with torch.no_grad():
            for imgs, lbls in val_loader:
                imgs, lbls = imgs.to(device), lbls.to(device)
                with torch.amp.autocast('cuda'):
                    out = ema_model(imgs)
                correct   += (out.argmax(1) == lbls).sum().item()
                total_val += lbls.size(0)

        val_acc  = correct / total_val
        avg_loss = total_loss / total
        print(f"  Loss: {avg_loss:.4f} | Val: {val_acc:.4f} | Best: {best_acc:.4f}")

        # Save immediately on every improvement
        if val_acc > best_acc:
            best_acc = val_acc
            torch.save({
                "state_dict":  ema_model.state_dict(),
                "best_acc":    best_acc,
                "num_classes": num_classes,
            }, SAVE_PATH)
            print(f"  ⭐ Saved! New best: {best_acc:.4f}")

    # ── Inference ─────────────────────────────────────────────────
    print(f"\n✅ Loading best model (acc={best_acc:.4f})")
    ckpt = torch.load(SAVE_PATH, map_location=device)
    ema_model.load_state_dict(ckpt["state_dict"])
    ema_model.eval()

    test_files      = sorted(glob.glob(os.path.join(TEST_DIR, "*.*")))
    results         = []
    corrupted_count = 0

    print(f"📸 Inference on {len(test_files)} images with 4-TTA...")
    with torch.no_grad():
        for path in tqdm(test_files):
            fname = os.path.basename(path)
            try:
                img    = Image.open(path).convert("RGB")
                logits = 0
                for t in TTA_TRANSFORMS:
                    inp = t(img).unsqueeze(0).to(device)
                    with torch.amp.autocast('cuda'):
                        logits += torch.softmax(ema_model(inp), dim=1)
                pred = logits.argmax(1).item()
                results.append({"ImageName": fname, "label": pred})
            except Exception as e:
                corrupted_count += 1
                print(f"⚠️  Corrupted: {fname} → label {dominant_class}")
                results.append({"ImageName": fname, "label": dominant_class})

    if corrupted_count:
        print(f"⚠️  {corrupted_count} corrupted images used fallback label")
    else:
        print("✅ No corrupted images")

    pd.DataFrame(results).to_csv("submission.csv", index=False)
    print(f"\n✅ submission.csv saved!")
    print(f"🏆 Best val accuracy: {best_acc:.4f}")


if __name__ == '__main__':
    run()

🖥️  Device: cuda
📂 Train: /kaggle/input/competitions/cse-281-spring-26-scene-style-classification/StyleClassificationIndoors/StyleClassificationIndoors/train
📂 Test:  /kaggle/input/competitions/cse-281-spring-26-scene-style-classification/StyleClassificationIndoors/StyleClassificationIndoors/test
📊 17 classes | 13163 total images
   Train: 11846 | Val: 1317
Downloading: "https://download.pytorch.org/models/convnext_base-6075fbad.pth" to /root/.cache/torch/hub/checkpoints/convnext_base-6075fbad.pth


100%|██████████| 338M/338M [00:01<00:00, 186MB/s]



🔥 Training 13 epochs (~52 min)...


Ep 01/13: 100%|██████████| 741/741 [04:18<00:00,  2.87it/s]


  Loss: 2.4485 | Val: 0.1845 | Best: 0.0000
  ⭐ Saved! New best: 0.1845


...

⚠️  Corrupted: testimage_3427.jpg → label 1


 65%|██████▍   | 3537/5482 [05:50<03:30,  9.26it/s]

⚠️  Corrupted: testimage_4180.jpg → label 1


100%|██████████| 5482/5482 [08:59<00:00, 10.16it/s]

⚠️  4 corrupted images used fallback label

✅ submission.csv saved!
🏆 Best val accuracy: 0.5171
